[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heitorramos/icd/blob/main/exemplos/23-regularizacao/notebook-colab.ipynb)


In [ ]:
# Preparação automática para execução no Google Colab.
# Fora do Colab, esta célula não altera o diretório de trabalho.
try:
    import google.colab  # type: ignore
except ImportError:
    pass
else:
    import os
    import subprocess
    from pathlib import Path

    repository = Path("/content/icd")
    if not repository.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/heitorramos/icd.git", str(repository)
        ], check=True)
    os.chdir(repository / "exemplos/23-regularizacao")
    print("Material preparado em:", Path.cwd())


# Regularização: viés, variância e estabilidade

Material de apoio — Aula 23

## Objetivos

Este guia explica por que baixo erro de treinamento não garante
generalização, deriva Ridge e compara as penalizações $L_0$, $L_1$ e
$L_2$. O estudo numérico usa apenas NumPy para tornar explícitas as
etapas de padronização, ajuste e validação.

## Como estudar este capítulo

Um modelo pode aprender tão bem os detalhes da amostra de treinamento
que passa a reproduzir também seu ruído. Regularização responde a esse
problema acrescentando ao ajuste um custo para modelos excessivamente
complexos. Aceitamos um pouco mais de erro no treinamento em troca de
coeficientes mais estáveis e melhor desempenho em dados novos.

O parâmetro $\lambda$ controla essa troca: quando é pequeno, a
prioridade permanece no ajuste; quando cresce, a penalização ganha
força. Ridge reduz continuamente os coeficientes, enquanto Lasso pode
levar alguns exatamente a zero. Nenhuma delas torna automaticamente o
modelo bom: representação, divisão dos dados e validação continuam
essenciais.

Preste atenção especial à ordem do processo. A padronização deve ser
aprendida somente com o conjunto de treinamento de cada divisão;
$\lambda$ deve ser escolhido na validação; o teste deve ser usado uma
única vez para estimar o desempenho final. Essa disciplina evita que
informação futura influencie o modelo.

## Base de dados de apoio

Usaremos [Medical Insurance
Cost](https://www.kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset),
do Kaggle. Cada linha representa uma pessoa segurada; `charges` registra
despesas e os demais campos descrevem idade, IMC, tabagismo, sexo,
filhos e região.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(20260827)
df = pd.read_csv(Path("../16-correlacao/data/insurance.csv"))
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

> **Interpretação**
>
> A base contém 1.338 observações sem valores ausentes. `charges` é
> assimétrica e tabagismo separa patamares de despesas. O objetivo é
> pedagógico e preditivo: coeficientes não devem ser interpretados como
> efeitos causais.

## Diagnóstico descritivo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=.45, ax=axes[0])
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker", alpha=.45, ax=axes[1], legend=False)
plt.tight_layout(); plt.show()

> **Interpretação**
>
> Idade e IMC têm associações com despesas, mas o padrão depende
> fortemente de tabagismo. Interações e termos não lineares podem
> melhorar aproximação; criar muitos deles, porém, aumenta
> instabilidade.

## Erro de treino e risco de generalização

Para uma regra $f$, o erro empírico quadrático é

$$
\widehat R_{treino}(f)=\frac1n\sum_{i=1}^n(y_i-f(x_i))^2.
$$

O objetivo real é o risco sobre uma nova observação da mesma população:

$$R(f)=E[(Y-f(X))^2].$$

$n$ é o tamanho do treino, $(x_i,y_i)$ é a observação $i$ e a esperança
em $R(f)$ abrange novos conjuntos de treino e novas observações. Modelos
mais flexíveis tendem a reduzir $\widehat R_{treino}$, mas não
necessariamente $R$.

## Decomposição viés–variância

Suponha $Y=f(x)+\varepsilon$, com $E[\varepsilon]=0$ e
$\operatorname{Var}(\varepsilon)=\sigma^2$. Em um ponto fixo $x$,

$$E[(Y-\widehat f(x))^2]
=\sigma^2+\{E[\widehat f(x)]-f(x)\}^2
+E[(\widehat f(x)-E[\widehat f(x)])^2].$$

Os termos são, respectivamente, ruído irredutível, viés ao quadrado e
variância do procedimento de ajuste.

Para derivar, escreva

$$Y-\widehat f=(f+\varepsilon-E[\widehat f])+(E[\widehat f]-\widehat f).$$

Ao expandir o quadrado, os termos cruzados somem porque
$E[\varepsilon]=0$ e $E[\widehat f-E\widehat f]=0$.

> **Interpretação**
>
> Regularização deliberadamente aumenta viés ao contrair coeficientes. A
> aposta é reduzir mais a variância, diminuindo o erro total fora da
> amostra.

## Construção de atributos

Criaremos termos polinomiais e interações. Essa expansão é
propositalmente redundante para revelar o efeito da regularização.

In [ ]:
age = df["age"].to_numpy(float)
bmi = df["bmi"].to_numpy(float)
smoker = (df["smoker"] == "yes").to_numpy(float)
y = df["charges"].to_numpy(float)

X_raw = np.column_stack([
    age, bmi, smoker, age*bmi, age*smoker, bmi*smoker,
    age**2, bmi**2, age**3, bmi**3
])
names = ["age", "bmi", "smoker", "age:bmi", "age:smoker",
         "bmi:smoker", "age²", "bmi²", "age³", "bmi³"]
pd.DataFrame(X_raw, columns=names).head()

## Divisão e padronização sem vazamento

In [ ]:
idx = rng.permutation(len(df))
n_train = int(.60*len(df))
n_valid = int(.20*len(df))
train = idx[:n_train]
valid = idx[n_train:n_train+n_valid]
test = idx[n_train+n_valid:]

mean_x = X_raw[train].mean(axis=0)
sd_x = X_raw[train].std(axis=0)
X_train = (X_raw[train]-mean_x)/sd_x
X_valid = (X_raw[valid]-mean_x)/sd_x
X_test = (X_raw[test]-mean_x)/sd_x
y_mean = y[train].mean()
y_train = y[train]-y_mean

pd.Series({"n_treino": len(train), "n_validacao": len(valid), "n_teste": len(test),
           "p": X_train.shape[1]})

> **Interpretação**
>
> Médias e desvios foram aprendidos apenas no treino e reaplicados ao
> teste. Padronizar antes da divisão permitiria que o teste
> influenciasse o treinamento. Como $y$ foi centrado, o intercepto é
> recuperado por `y_mean` e não será penalizado.

## Formulação geral

Regularização resolve

$$
\widehat\beta_\lambda=\arg\min_\beta
\left\{\frac1n\|y-X\beta\|_2^2+\lambda P(\beta)\right\}.
$$

$X$ é a matriz padronizada, $y$ é a resposta centrada, $P(\beta)$ mede
complexidade e $\lambda\ge0$ é a intensidade da penalização.

As escolhas principais são:

$$\|\beta\|_0=\sum_j\mathbb 1(\beta_j\ne0),\qquad
\|\beta\|_1=\sum_j|\beta_j|,\qquad
\|\beta\|_2^2=\sum_j\beta_j^2.$$

$L_0$ conta termos e exige busca combinatória; $L_1$ produz Lasso; $L_2$
produz Ridge.

## Derivação de Ridge

Para

$$J(\beta)=\frac1n(y-X\beta)^T(y-X\beta)+\lambda\beta^T\beta,$$

o gradiente é

$$\nabla J(\beta)=-\frac2nX^T(y-X\beta)+2\lambda\beta.$$

Igualando a zero:

$$X^TX\beta+n\lambda\beta=X^Ty,$$

portanto

$$\widehat\beta^{ridge}=(X^TX+n\lambda I)^{-1}X^Ty.$$

$I$ é a identidade $p\times p$. Adicionar $n\lambda I$ eleva os
autovalores da matriz, melhorando o condicionamento.

## Ajuste Ridge

In [ ]:
def ridge(X, y, lam):
    p = X.shape[1]
    return np.linalg.solve(X.T@X + len(X)*lam*np.eye(p), X.T@y)

def rmse(observed, predicted):
    return np.sqrt(np.mean((observed-predicted)**2))

beta_ols = np.linalg.lstsq(X_train, y_train, rcond=None)[0]
beta_ridge = ridge(X_train, y_train, lam=.01)
pd.DataFrame({"OLS": beta_ols, "Ridge": beta_ridge}, index=names).round(2)

> **Interpretação**
>
> Os coeficientes estão na escala padronizada dos atributos, mas
> `charges` permanece em dólares. Ridge contrai sobretudo combinações
> redundantes de termos lineares, polinomiais e interações. O valor
> isolado de cada coeficiente depende da base expandida.

## Caminho de regularização e validação

In [ ]:
lambdas = np.logspace(-6, 2, 100)
betas = np.array([ridge(X_train, y_train, lam) for lam in lambdas])
train_rmse = np.array([rmse(y[train], y_mean+X_train@b) for b in betas])
valid_rmse = np.array([rmse(y[valid], y_mean+X_valid@b) for b in betas])
best = np.argmin(valid_rmse)

pd.Series({"lambda_demonstrativo": lambdas[best],
           "RMSE_treino": train_rmse[best],
           "RMSE_validacao": valid_rmse[best]}).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for j, name in enumerate(names):
    axes[0].semilogx(lambdas, betas[:, j], label=name)
axes[0].axhline(0, color="black", linewidth=.8)
axes[0].set(xlabel="lambda", ylabel="coeficiente", title="Caminho Ridge")
axes[1].semilogx(lambdas, train_rmse, label="treino")
axes[1].semilogx(lambdas, valid_rmse, label="validação")
axes[1].set(xlabel="lambda", ylabel="RMSE", title="Compromisso de generalização")
axes[1].legend(); plt.tight_layout(); plt.show()

> **Interpretação**
>
> O conjunto de validação escolhe $\lambda$ sem consultar o teste. O
> teste permanece reservado para estimar o desempenho final depois que
> todas as decisões foram tomadas.

## Avaliação final no teste

In [ ]:
beta_selected = betas[best]
pd.Series({
    "lambda_escolhido": lambdas[best],
    "RMSE_validacao": valid_rmse[best],
    "RMSE_teste_final": rmse(y[test], y_mean+X_test@beta_selected),
}).round(3)

> **Interpretação**
>
> A diferença entre validação e teste lembra que a estimativa de
> desempenho também varia entre amostras. O teste não retorna para
> alterar $\lambda$; isso preserva sua função de avaliação final.

## Lasso e o operador de limiarização

No caso ortogonal e padronizado, a solução Lasso aplica *soft
thresholding* à solução não penalizada:

$$S(z,\gamma)=\operatorname{sign}(z)(|z|-\gamma)_+,$$

em que $(a)_+=\max(a,0)$. Se $|z|\le\gamma$, o coeficiente se torna
exatamente zero.

In [ ]:
def soft_threshold(z, gamma):
    return np.sign(z)*np.maximum(np.abs(z)-gamma, 0)

z = np.array([-2., -.4, .2, 1.5])
pd.DataFrame({"z": z, "S(z, 0.5)": soft_threshold(z, .5)})

> **Interpretação**
>
> O intervalo em torno de zero é colapsado para zero. Essa é a origem
> algorítmica da esparsidade do Lasso. Com atributos correlacionados, a
> solução requer otimização iterativa e a seleção pode variar entre
> amostras.

## Elastic Net

Elastic Net usa

$$P(\beta)=\alpha\|\beta\|_1+\frac{1-\alpha}{2}\|\beta\|_2^2.$$

$\lambda$ controla a força total; $\alpha$ controla a mistura. A parcela
$L_1$ permite zeros e a parcela $L_2$ estabiliza grupos de preditores
correlacionados.

## Regularização na regressão logística

Na Aula 22, minimizamos a entropia cruzada. Com Ridge logístico:

$$J(\beta)=-\frac1n\sum_{i=1}^n[y_i\log\pi_i+(1-y_i)\log(1-\pi_i)]
+\lambda\sum_{j=1}^p\beta_j^2.$$

$\pi_i=\sigma(x_i^T\beta)$ é a probabilidade prevista. O gradiente
acrescenta $2\lambda\beta$ ao gradiente da log-loss. O intercepto
continua excluído da penalização.

> **Interpretação**
>
> Na separação perfeita, a verossimilhança logística pode favorecer
> coeficientes com magnitude crescente. A penalização L2 fornece uma
> solução finita e mais estável.

## Limitações e inferência

Regularização não corrige variáveis omitidas, mudança de distribuição,
rótulos enviesados ou vazamento. Ela produz estimativas deliberadamente
enviesadas; intervalos e testes clássicos de OLS não devem ser
reutilizados ingenuamente após seleção ou penalização.

Para previsão, reporte validação e desempenho final. Para inferência,
descreva a seleção e use um procedimento compatível com ela.

## Regularização passo a passo

Regularizar não significa simplesmente “diminuir coeficientes”. O
procedimento completo é:

1.  **Construir uma representação candidata.** Ela pode conter muitos
    atributos, transformações e termos correlacionados.
2.  **Separar desenvolvimento e teste.** O teste não participa da
    escolha da intensidade de regularização.
3.  **Padronizar dentro da pipeline.** Sem escala comparável, a
    penalidade trata unidades diferentes de modo desigual.
4.  **Escolher uma família.** Ridge reduz todos os coeficientes; Lasso
    também pode zerar alguns; Elastic Net combina os dois
    comportamentos.
5.  **Testar vários valores de $\lambda$.** Valores pequenos se
    aproximam do modelo não regularizado; valores grandes simplificam
    mais.
6.  **Escolher por validação.** Comparamos desempenho médio e
    estabilidade, não apenas erro de treinamento.
7.  **Reajustar e avaliar uma vez no teste.** Essa avaliação representa
    a pipeline pronta.

## Como entender Ridge e Lasso

Ridge é especialmente útil quando muitos atributos carregam informação
parecida. Em vez de permitir coeficientes grandes e instáveis, distribui
a contribuição de forma mais suave.

Lasso pode gerar um modelo mais curto, mas “coeficiente zero” não prova
que a variável seja cientificamente irrelevante. Com atributos muito
correlacionados, pequenas mudanças nos dados podem fazer o método
escolher um representante diferente.

Elastic Net costuma ser uma alternativa quando queremos alguma seleção
sem abandonar completamente grupos de atributos correlacionados.

## O que observar nas saídas

- curva de validação: onde o erro deixa de melhorar de forma relevante;
- caminho dos coeficientes: quais termos encolhem cedo e quais
  permanecem;
- diferença entre treino e validação: indício do compromisso
  viés–variância;
- estabilidade entre folds ou reamostragens;
- desempenho final e padrão de erros no teste.

> **Interpretação**
>
> O melhor $\lambda$ não é uma constante da base. Ele depende dos
> atributos, da padronização, da métrica, das divisões e do modelo. Por
> isso toda a pipeline deve ser validada em conjunto.

## Síntese

- baixo erro de treino não garante baixo risco;
- Ridge reduz variância e estabiliza colinearidade;
- Lasso contrai e pode selecionar atributos;
- Elastic Net combina seleção e estabilidade;
- padronização deve ocorrer dentro da validação;
- $\lambda$ é escolhido sem consultar o teste final.

## Bibliografia

- James et al., *An Introduction to Statistical Learning*, capítulo 6.
- Hastie, Tibshirani e Friedman, *The Elements of Statistical Learning*,
  capítulos 3 e 7.
- Deisenroth, Faisal e Ong, *Mathematics for Machine Learning*, capítulo
  9.
- Data 100, capítulos sobre engenharia de atributos e regularização.